# Description

In this notebook, I will test the inference of GPT-2 model

In [1]:
import os
from pathlib import Path
import zipfile
import math
from datasets import load_dataset
from collections import OrderedDict
import re 

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import tiktoken
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken

/home/tnguyen10/.conda/envs/gpu_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# TODO: Replace with actual int8 matmul implementation
def dummy_int8_matmul(A_int8: torch.Tensor, B_int: torch.Tensor, out_dtype=torch.int32):
    """
    This is a dummy int8 matrix multiplication function.
    """
    if A_int8.dtype != torch.int8 or B_int.dtype != torch.int8:
        raise ValueError("Both A and B must be int8 tensors.")
    result_float = torch.matmul(A_int8.float(), B_int.float())
    return result_float.to(out_dtype)


def quantized_column_matrix_int_symmetric(mat:torch.Tensor):
    """
    Symmetric quantization to int8 on a per-column basis.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    qmin = -128
    qmax = 127
    
    max_vals, _ = torch.max(torch.abs(mat), dim=0, keepdim=True)  # shape (1, M)
    scales = (max_vals / qmax).squeeze(0)  # shape (M,)
    
    q_mat = torch.clamp(torch.round(mat / scales.unsqueeze(0)), qmin, qmax).to(torch.int8)  # shape (N, M)
    
    scales = scales.clone().detach().to(torch.float32)
    return q_mat, scales

def quantize_row_matrix_int8_symmetric_batched(mat: torch.Tensor):
    """
    Symmetric per-row quantization for batched 3D tensor.
    mat: [B, N, D]  (float tensor)
    
    Returns:
        q_mat:   [B, N, D] int8
        scales:  [B, N]    float32  (scale per row within each batch)
    """
    qmin, qmax = -128, 127

    # Compute max abs per row (per batch) - Result shape: [B, N, 1]
    max_vals, _ = torch.max(torch.abs(mat), dim=2, keepdim=True)

    # Compute scales per row
    scales = (max_vals / qmax).clamp(min=1e-12)  # avoid div-by-zero, shape [B, N, 1]

    # Quantize
    q_mat = torch.clamp(torch.round(mat / scales), qmin, qmax).to(torch.int8)

    # Return float scales of shape [B, N]
    scales = scales.squeeze(2).to(torch.float32)
    return q_mat, scales      


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by n_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Parameter(torch.empty(d_in, d_out))
        self.W_query_bias = nn.Parameter(torch.empty(d_out))
        
        self.W_key = nn.Parameter(torch.empty(d_in, d_out))
        self.W_key_bias = nn.Parameter(torch.empty(d_out))
        
        self.W_value = nn.Parameter(torch.empty(d_in, d_out))
        self.W_value_bias = nn.Parameter(torch.empty(d_out))
        
        self.out_proj = nn.Parameter(torch.empty(d_out, d_out))
        self.out_proj_bias = nn.Parameter(torch.empty(d_out))
        
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
        
        self.is_merge_w_and_b = False
        self.register_buffer("W_q", torch.empty(d_in + 1, d_out), persistent=False)
        self.register_buffer("W_k", torch.empty(d_in + 1, d_out), persistent=False)
        self.register_buffer("W_v", torch.empty(d_in + 1, d_out), persistent=False)
        self.register_buffer("out", torch.empty(d_out + 1, d_out), persistent=False)
        
        # Quantization parameters
        self.register_buffer("W_q_q", torch.empty_like(self.W_q, dtype=torch.int8), persistent=False)
        self.register_buffer("W_k_q", torch.empty_like(self.W_k, dtype=torch.int8), persistent=False)
        self.register_buffer("W_v_q", torch.empty_like(self.W_v, dtype=torch.int8), persistent=False)
        self.register_buffer("out_q", torch.empty_like(self.out, dtype=torch.int8), persistent=False)
        
        self.register_buffer("W_q_scale", torch.empty(d_out, dtype=torch.float32), persistent=False)
        self.register_buffer("W_k_scale", torch.empty(d_out, dtype=torch.float32), persistent=False)
        self.register_buffer("W_v_scale", torch.empty(d_out, dtype=torch.float32), persistent=False)
        self.register_buffer("out_scale", torch.empty(d_out, dtype=torch.float32), persistent=False)
        
        self.is_quantized = False
        
    @torch.no_grad()
    def merge_weight_bias(self):        
        W_q = torch.concat([self.W_query, self.W_query_bias.unsqueeze(0)], dim=0)
        W_k = torch.concat([self.W_key, self.W_key_bias.unsqueeze(0)], dim=0)
        W_v = torch.concat([self.W_value, self.W_value_bias.unsqueeze(0)], dim=0)
        out = torch.concat([self.out_proj, self.out_proj_bias.unsqueeze(0)], dim=0)

        self.W_q.copy_(W_q)
        self.W_k.copy_(W_k)
        self.W_v.copy_(W_v)
        self.out.copy_(out)
        
        self.is_merge_w_and_b = True
        
    @torch.no_grad()
    def quantize_weights(self):
        if not self.is_merge_w_and_b:
            self.merge_weight_bias()
        
        W_q_q, W_q_scale = quantized_column_matrix_int_symmetric(self.W_q)
        W_k_q, W_k_scale = quantized_column_matrix_int_symmetric(self.W_k)
        W_v_q, W_v_scale = quantized_column_matrix_int_symmetric(self.W_v)
        out_q, out_scale = quantized_column_matrix_int_symmetric(self.out)
        
        self.W_q_q.copy_(W_q_q)
        self.W_k_q.copy_(W_k_q)
        self.W_v_q.copy_(W_v_q)
        self.out_q.copy_(out_q)
        
        self.W_q_scale.copy_(W_q_scale)
        self.W_k_scale.copy_(W_k_scale)
        self.W_v_scale.copy_(W_v_scale)
        self.out_scale.copy_(out_scale)
        print(f"[INFO] Done quantizing current layer weights.")
        
        self.is_quantized = True
        

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        if self.is_merge_w_and_b:
            x = torch.cat([x, torch.ones(b, num_tokens, 1, device=x.device)], dim=2)  # Add bias term
            
            if self.is_quantized:
                x_q, x_scale = quantize_row_matrix_int8_symmetric_batched(x)

                queries = dummy_int8_matmul(x_q, self.W_q_q)
                keys = dummy_int8_matmul(x_q, self.W_k_q)
                values_int = dummy_int8_matmul(x_q, self.W_v_q)

                global_wq_scale = self.W_q_scale.mean()
                global_wk_scale = self.W_v_scale.mean()
            
                # scale_of_q = x_scale.unsqueeze(-1) * self.W_q_scale[None, :]
                scale_of_q = x_scale.unsqueeze(-1) * global_wq_scale
                queries = queries * scale_of_q
                queries = queries.to(x.dtype)
                
                # scale_of_k = x_scale.unsqueeze(-1) * self.W_k_scale[None, :]
                scale_of_k = x_scale.unsqueeze(-1) * global_wk_scale
                keys = keys * scale_of_k
                keys = keys.to(x.dtype)
                
                values = x_scale.unsqueeze(-1) * values_int * self.W_v_scale[None, :]
                values = values.to(x.dtype)
            
            else: # normal float matmul
                queries = x @ self.W_q  # Shape: (b, num_tokens, d_out)
                keys = x @ self.W_k  # Shape: (b, num_tokens, d_out)
                values = x @ self.W_v  # Shape: (b, num_tokens, d_out)
            
        else:
            queries = x @ self.W_query + self.W_query_bias  # Shape: (b, num_tokens, d_out)
            keys = x @ self.W_key + self.W_key_bias  # Shape: (b, num_tokens, d_out)
            values = x @ self.W_value + self.W_value_bias  # Shape: (b, num_tokens, d_out)

        # Reshape to split heads: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) 
        if self.is_quantized:
            attn_scores = queries @ keys.transpose(2, 3)  
        else:
            attn_scores = queries @ keys.transpose(2, 3) 

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        if self.is_merge_w_and_b:
            context_vec = torch.cat([context_vec, torch.ones(b, num_tokens, 1, device=x.device)], dim=2)
            context_vec = context_vec @ self.out
        else:
            context_vec = context_vec @ self.out_proj + self.out_proj_bias

        return context_vec


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_resid = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_resid(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_resid(x)
        x = x + shortcut  # Add the original input back

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Get the idx of the vocab entry with the highest logits value
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx


In [14]:
GPT_CONFIG_BASE = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Original context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.0,       # Dropout rate
    "qkv_bias": True        # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

PATH_MODEL = "/scratch/tnguyen10/gpt2-xl-1558M.pth"
# PATH_MODEL = "/scratch/tnguyen10/gpt2-medium-355M.pth"
model_name = "gpt2-xl (1558M)"  # FIX When changing model, update PATH_MODEL accordingly
# model_name = "gpt2-medium (355M)"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NEW_CONFIG = GPT_CONFIG_BASE.copy()
NEW_CONFIG.update(model_configs[model_name])

model = GPTModel(NEW_CONFIG)
model.to(DEVICE)

print(f"Done loading model {model_name}.")

Done loading model gpt2-xl (1558M).


In [15]:
def remap_for_parameter_only(ckpt, target_dtype=None, device=None):
    """
    Remap state_dict from nn.Linear-style keys (...W_key.weight) with shape [out,in]
    to parameter-only keys (...W_key) with shape [in,out].
    Same for W_value, W_query, out_proj.
    """
    new_sd = OrderedDict()
    # patterns: replace ".W_key.weight" -> ".W_key" etc.
    patterns = [
        (re.compile(r"\.W_key\.weight$"),   ".W_key"),
        (re.compile(r"\.W_key\.bias$"),   ".W_key_bias"),
        (re.compile(r"\.W_value\.weight$"), ".W_value"),
        (re.compile(r"\.W_value\.bias$"), ".W_value_bias"),
        (re.compile(r"\.W_query\.weight$"), ".W_query"),
        (re.compile(r"\.W_query\.bias$"), ".W_query_bias"),
        (re.compile(r"\.out_proj\.weight$"), ".out_proj"),
        (re.compile(r"\.out_proj\.bias$"), ".out_proj_bias"),
    ]

    for k, v in ckpt.items():
        new_key = k
        for rgx, repl in patterns:
            if rgx.search(k):
                new_key = rgx.sub(repl, k)  # rename to parameter-only key
                v = v.t()                   # transpose [out,in] -> [in,out]  - This is just a nn.Linear torch convention
                break

        # cast if requested
        if target_dtype is not None:
            v = v.to(target_dtype)
        if device is not None:
            v = v.to(device)

        new_sd[new_key] = v

    return new_sd

checkpoint = torch.load(PATH_MODEL, weights_only=True, map_location=DEVICE)
checkpoint = remap_for_parameter_only(checkpoint, device=DEVICE)

model.load_state_dict(checkpoint)

<All keys matched successfully>

In [16]:
for name, module in model.named_modules():
    if isinstance(module, MultiHeadAttention):
        module.merge_weight_bias()

In [17]:
tokenizer = tiktoken.get_encoding("gpt2")
print(f"Tokenizer vocab size: {tokenizer.n_vocab}")

Tokenizer vocab size: 50257


In [18]:
prompt = "What is the capital of France?"
enc_prompt = tokenizer.encode(prompt)
enc_prompt = torch.tensor([enc_prompt])
enc_prompt = enc_prompt.to("cuda")

token_ids = generate_text_simple(
    model=model,
    idx=enc_prompt, 
    max_new_tokens=100, 
    context_size=NEW_CONFIG["context_length"]
)

output = tokenizer.decode(token_ids.squeeze().tolist())
print(output)

What is the capital of France?

The capital of France is Paris.

What is the capital of the United States?

The capital of the United States is Washington, D.C.

What is the capital of the United Kingdom?

The capital of the United Kingdom is London.

What is the capital of Canada?

The capital of Canada is Ottawa.

What is the capital of Australia?

The capital of Australia is Canberra.

What is the capital of


# 1. Quantize model

In [19]:
# Loop over each GroupedQueryAttention and perform quantization
for name, module in model.named_modules():
    if isinstance(module, MultiHeadAttention):
        print(f"\nQuantizing weights for module: {name} ...")
        module.quantize_weights()
        print(f"Quantization done for module: {name}.")


Quantizing weights for module: trf_blocks.0.att ...
[INFO] Done quantizing current layer weights.
Quantization done for module: trf_blocks.0.att.

Quantizing weights for module: trf_blocks.1.att ...
[INFO] Done quantizing current layer weights.
Quantization done for module: trf_blocks.1.att.

Quantizing weights for module: trf_blocks.2.att ...
[INFO] Done quantizing current layer weights.
Quantization done for module: trf_blocks.2.att.

Quantizing weights for module: trf_blocks.3.att ...
[INFO] Done quantizing current layer weights.
Quantization done for module: trf_blocks.3.att.

Quantizing weights for module: trf_blocks.4.att ...
[INFO] Done quantizing current layer weights.
Quantization done for module: trf_blocks.4.att.

Quantizing weights for module: trf_blocks.5.att ...
[INFO] Done quantizing current layer weights.
Quantization done for module: trf_blocks.5.att.

Quantizing weights for module: trf_blocks.6.att ...
[INFO] Done quantizing current layer weights.
Quantization done f

In [20]:
prompt = "What is the capital of France?"
enc_prompt = tokenizer.encode(prompt)
enc_prompt = torch.tensor([enc_prompt])
enc_prompt = enc_prompt.to("cuda")

token_ids = generate_text_simple(
    model=model,
    idx=enc_prompt, 
    max_new_tokens=100, 
    context_size=NEW_CONFIG["context_length"]
)

output = tokenizer.decode(token_ids.squeeze().tolist())
print(output)

What is the capital of France?

The capital of France is Paris.

What is the capital of the United States?

The capital of the United States is Washington.

What is the capital of the United Kingdom of Great Britain?

The capital of Great Britain is London.

What is the capital of the United States of America?

The capital of the United States of America is New York.

What is the capital of the Republic of the Philippines?

The capital of


# 2. Measure PPL

In [10]:
EOS_ID = 50256  # gpt2's end token (not strictly needed here)

@torch.no_grad()
def compute_ppl(model, tokenizer, texts, context_size, device=DEVICE):
    """
    Faster PPL: slide windows of length <= context_size and
    score only the last token of each window (which has full left context).
    """
    model_was_training = model.training
    model.eval()

    results = []
    for txt in texts:
        ids = tokenizer.encode(txt)
        if len(ids) < 2:
            results.append({"num_tokens": 0, "nll_sum": 0.0, "ppl": float("nan")})
            continue

        ids_t = torch.tensor(ids, dtype=torch.long, device=device)
        nll_sum = 0.0
        tok_cnt = 0

        # We will take windows ending at positions end=1..L-1
        L = ids_t.size(0)
        end = 1
        while end < L:
            start = max(0, end - context_size)        # include up to token end-1
            inp = ids_t[start:end].unsqueeze(0)       # [1, w] (predict token at 'end')
            logits = model(inp)                        # [1, w, V]
            last_logits = logits[:, -1, :]            # prediction for token at 'end'
            target = ids_t[end].view(1)               # [1]
            loss = F.cross_entropy(last_logits, target, reduction="sum")
            nll_sum += float(loss.item())
            tok_cnt += 1

            # Jump ahead by a stride: score roughly one token per window
            # (Tune stride for speed/accuracy trade-off; 1 is exact; larger is faster.)
            stride = max(1, context_size - 1)
            end += stride

        # If we skipped some tail tokens due to large stride, optionally finish them:
        if end - (context_size - 1) < L - 1:
            # exact tail sweep to ensure full coverage
            for t in range(max(1, L - context_size + 1), L):
                start = max(0, t - context_size)
                inp = ids_t[start:t].unsqueeze(0)
                logits = model(inp)
                last_logits = logits[:, -1, :]
                target = ids_t[t].view(1)
                loss = F.cross_entropy(last_logits, target, reduction="sum")
                nll_sum += float(loss.item())
                tok_cnt += 1

        ppl = math.exp(nll_sum / max(tok_cnt, 1))
        results.append({"num_tokens": tok_cnt, "nll_sum": nll_sum, "ppl": ppl})

    total_nll = sum(r["nll_sum"] for r in results)
    total_tok = sum(r["num_tokens"] for r in results) or 1
    corpus_ppl = math.exp(total_nll / total_tok)

    if model_was_training: model.train()
    return results, corpus_ppl

In [11]:
# ----- Load 1,000 samples from WikiText2 -----
def load_wikitext2_samples(n=1000, min_length=10):
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
    # Filter out empty or too-short lines
    samples = [x["text"] for x in dataset if len(x["text"].strip()) > min_length]
    return samples[:n]

num_samples = 10

samples = load_wikitext2_samples(num_samples)
print(f"Loaded {len(samples)} samples. Computing perplexity...")

Loaded 10 samples. Computing perplexity...


In [12]:
per_text, corpus_ppl = compute_ppl(
    model=model,
    tokenizer=tokenizer,           
    texts=samples,
    context_size=NEW_CONFIG["context_length"],
    device=DEVICE
)

# print(per_text)
print("Corpus PPL:", corpus_ppl)

Corpus PPL: 26.25210067834266
